In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from adjustText import adjust_text
import warnings
from scipy.stats import norm
warnings.filterwarnings('ignore')


# Read in data

In [ ]:
gp2_female_df = pd.read_csv(f'{WORK_DIR}/GP2/GP2.EUR.FEMALE.combined.final.FOR_PLINK_freq.txt', sep='\s+') 
gp2_female_df['new_name'] = 'chr' + gp2_female_df['CHR'].astype(str) + ':' + gp2_female_df['BP'].astype('str')
gp2_female_df.head()
                              

In [ ]:
gp2_male_df = pd.read_csv(f'{WORK_DIR}/GP2/GP2.EUR.MALE.combined.final.FOR_PLINK_freq.txt', sep='\s+') 
gp2_male_df['new_name'] = 'chr' + gp2_male_df['CHR'].astype(str) + ':' + gp2_male_df['BP'].astype('str')
gp2_male_df.head()
                              

In [50]:
df = pd.read_csv("/data/CARD_AA/projects/2025_01_HL_PD_MALE_FEMALE/mary/gp2_vars_v2.txt", sep="\t")
gp2_vars = df["SNP"].tolist()

In [51]:
len(gp2_vars)

157

In [ ]:
all_female = pd.read_csv(f'{WORK_DIR}/METAS/FINAL/ALL_FEMALE_COMBINED.meta', delim_whitespace=True)
all_female['SE'] = abs(all_female['BETA']/(norm.ppf(all_female['P']/2)))

all_male = pd.read_csv(f'{WORK_DIR}/METAS/FINAL/ALL_MALE_COMBINED.meta', delim_whitespace=True)
all_male['SE'] = abs(all_male['BETA']/(norm.ppf(all_male['P']/2)))

# Plot

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from adjustText import adjust_text
import warnings
warnings.filterwarnings('ignore')

def investigate_duplicates(df, cohort_name):
    """
    Investigate duplicate variants before dropping them
    """
    print(f"\n{'='*80}")
    print(f"INVESTIGATING DUPLICATES IN: {cohort_name}")
    print(f"{'='*80}\n")
    
    # Check for duplicates on SNP column
    if 'SNP' not in df.columns:
        print("No SNP column found")
        return None
    
    # Find duplicates
    dupes = df[df['SNP'].duplicated(keep=False)].copy()
    
    if len(dupes) == 0:
        print("✅ No duplicates found!")
        return None
    
    print(f"Found {len(dupes):,} duplicate rows")
    print(f"Affecting {dupes['SNP'].nunique():,} unique variants\n")
    
    # Show duplicate counts
    dupe_counts = dupes['SNP'].value_counts()
    print("Duplicate frequency:")
    print(f"  2 copies: {(dupe_counts == 2).sum():,}")
    print(f"  3 copies: {(dupe_counts == 3).sum():,}")
    print(f"  4+ copies: {(dupe_counts >= 4).sum():,}")
    
    # Examine first few duplicates
    print(f"\n📋 EXAMPLE DUPLICATES:\n")
    for i, (var_id, count) in enumerate(dupe_counts.head(5).items()):
        print(f"\n{i+1}. {var_id} ({count} copies):")
        var_data = dupes[dupes['SNP'] == var_id]
        
        cols_to_show = ['SNP', 'CHR', 'BP', 'A1', 'A2', 'BETA', 'SE', 'P', 'EAF']
        cols_to_show = [c for c in cols_to_show if c in var_data.columns]
        
        print(var_data[cols_to_show].to_string(index=False))
        
        # Check if they differ
        if 'P' in var_data.columns:
            p_diff = var_data['P'].max() - var_data['P'].min()
            if p_diff > 0.01:
                print(f"  P-values differ substantially (range: {p_diff:.2e})")
        
        if 'BETA' in var_data.columns:
            beta_diff = var_data['BETA'].max() - var_data['BETA'].min()
            if beta_diff > 0.01:
                print(f"  BETA values differ substantially (range: {beta_diff:.4f})")
    
    return dupes


def load_and_prep_gwas(df_or_path, cohort_name, snplist=None, drop_duplicates=True, return_dropped=False):
    # Load data if path provided
    if isinstance(df_or_path, str):
        try:
            df = pd.read_csv(df_or_path, sep='\t', low_memory=False)
        except:
            df = pd.read_csv(df_or_path, sep=',', low_memory=False)
    else:
        df = df_or_path.copy()
    
    print(f"Loaded {cohort_name}: {len(df):,} variants")
    
    # Filter to snplist FIRST if provided (MOVED THIS UP)
    if snplist is not None:
        if 'SNP' in df.columns:
            n_before_filter = len(df)
            df = df[df['SNP'].isin(snplist)].copy()
            print(f"  Filtered to snplist: {len(df):,} variants (from {n_before_filter:,})")
            
            # Report which SNPs from the list are missing
            found_snps = set(df['SNP'].unique())
            requested_snps = set(snplist)
            missing_snps = requested_snps - found_snps
            if missing_snps:
                print(f" Warning: {len(missing_snps)} SNPs from snplist not found in data")
                if len(missing_snps) <= 10:
                    print(f"     Missing: {list(missing_snps)}")
        else:
            print(f"  Warning: SNP column not found, cannot filter by snplist")
    
    dropped_dupes = None
    
    # Drop duplicates if requested (AFTER filtering to snplist)
    if drop_duplicates:
        if 'SNP' not in df.columns:
            print("  Warning: No SNP column found, cannot drop duplicates")
        else:
            n_before = len(df)
            
            # Identify duplicates before dropping
            if return_dropped:
                # Get all duplicate rows (including the one we'll keep)
                all_dupes = df[df['SNP'].duplicated(keep=False)].copy()
                
                # Sort by P-value to determine which will be kept
                if 'P' in df.columns:
                    df_sorted = df.sort_values('P')
                    kept_snps = df_sorted.drop_duplicates(subset='SNP', keep='first')['SNP']
                    # Dropped duplicates are all dupes except the ones we kept
                    dropped_dupes = all_dupes[~all_dupes['SNP'].isin(kept_snps) | 
                                               (all_dupes['SNP'].duplicated(keep='first'))].copy()
                else:
                    kept_snps = df.drop_duplicates(subset='SNP', keep='first')['SNP']
                    dropped_dupes = all_dupes[~all_dupes['SNP'].isin(kept_snps) | 
                                               (all_dupes['SNP'].duplicated(keep='first'))].copy()
            
            # Actually drop duplicates
            if 'P' in df.columns:
                df = df.sort_values('P').drop_duplicates(subset='SNP', keep='first')
            else:
                df = df.drop_duplicates(subset='SNP', keep='first')
            
            n_removed = n_before - len(df)
            if n_removed > 0:
                print(f"  Removed {n_removed:,} duplicates from {cohort_name} (kept lowest P-value)")
    
    # Add cohort label
    df['COHORT'] = cohort_name
    
    print(f"  Final dataset: {len(df):,} variants")
    
    if return_dropped:
        return df, dropped_dupes
    else:
        return df


def merge_gwas_for_comparison(df1, df2, label1, label2):
    """
    Merge two GWAS datasets for comparison on SNP column
    """
    if 'SNP' not in df1.columns or 'SNP' not in df2.columns:
        raise ValueError("SNP column not found in both dataframes")
    
    print(f"Merging on: SNP")
    
    # Merge on SNP
    merged = df1.merge(
        df2, 
        on='SNP', 
        how='inner',
        suffixes=('_1', '_2')
    )
    
    print(f"Merged datasets: {len(merged):,} overlapping variants")
    
    # Rename columns for clarity
    rename_map = {
        'BETA_1': f'BETA_{label1}',
        'BETA_2': f'BETA_{label2}',
        'SE_1': f'SE_{label1}',
        'SE_2': f'SE_{label2}',
        'P_1': f'P_{label1}',
        'P_2': f'P_{label2}',
        'A1_1': f'A1_{label1}',
        'A1_2': f'A1_{label2}',
        'A2_1': f'A2_{label1}',
        'A2_2': f'A2_{label2}',
        'EAF_1': f'EAF_{label1}',
        'EAF_2': f'EAF_{label2}',
    }
    
    # Only rename columns that exist
    rename_map = {k: v for k, v in rename_map.items() if k in merged.columns}
    merged.rename(columns=rename_map, inplace=True)
    
    return merged


def classify_significance(merged, label1, label2, sig_level=5e-8):
    """
    Classify variants by significance in each cohort
    """
    p1_col = f'P_{label1}'
    p2_col = f'P_{label2}'
    
    # Check if P-value columns exist
    if p1_col not in merged.columns or p2_col not in merged.columns:
        merged['category'] = 'None'
        return merged
    
    # Classify
    sig_1 = merged[p1_col] < sig_level
    sig_2 = merged[p2_col] < sig_level
    
    merged['category'] = 'None'
    merged.loc[sig_1 & sig_2, 'category'] = 'Both'
    merged.loc[sig_1 & ~sig_2, 'category'] = label1
    merged.loc[~sig_1 & sig_2, 'category'] = label2
    
    # Print summary
    print("\nSignificance classification:")
    print(merged['category'].value_counts())
    
    return merged


def calculate_concordance(merged, label1, label2):
    """
    Calculate concordance metrics between two GWAS
    """
    beta1_col = f'BETA_{label1}'
    beta2_col = f'BETA_{label2}'
    
    if beta1_col not in merged.columns or beta2_col not in merged.columns:
        return None
    
    # Remove missing values
    valid = merged[[beta1_col, beta2_col]].dropna()
    
    if len(valid) == 0:
        return None
    
    # Calculate correlation
    r, p = stats.pearsonr(valid[beta1_col], valid[beta2_col])
    
    # Calculate R²
    r2 = r ** 2
    
    # Linear regression
    slope, intercept, _, _, _ = stats.linregress(valid[beta1_col], valid[beta2_col])
    
    # Sign concordance (same direction)
    sign_concordance = (np.sign(valid[beta1_col]) == np.sign(valid[beta2_col])).mean()
    
    metrics = {
        'r': r,
        'r2': r2,
        'p_value': p,
        'slope': slope,
        'intercept': intercept,
        'sign_concordance': sign_concordance,
        'n': len(valid)
    }
    
    return metrics


def find_most_discordant(merged, label1, label2, n=5):
    """
    Find variants with most discordant effect sizes
    Returns list of variant identifiers
    """
    beta1_col = f'BETA_{label1}'
    beta2_col = f'BETA_{label2}'
    
    if beta1_col not in merged.columns or beta2_col not in merged.columns:
        return []
    
    # Calculate absolute difference
    merged['beta_diff'] = np.abs(merged[beta1_col] - merged[beta2_col])
    
    # Get top N most discordant
    most_discordant = merged.nlargest(n, 'beta_diff')
    
    # Return SNP identifiers
    if 'SNP' in merged.columns:
        return most_discordant['SNP'].tolist()
    else:
        return []


def create_beta_beta_plot(
    merged, 
    label1, 
    label2,
    labels=None,
    sig_level=5e-8,
    annotate_snps=None,
    annotate_discordant=0,
    plot_error_bars=True,
    color_map=None,
    alpha=0.6,
    s=50,
    figsize=(10, 10),
    save=None,
    dpi=300
):
    """
    Create beta-beta scatter plot with error bars and custom markers
    """
    beta1_col = f'BETA_{label1}'
    beta2_col = f'BETA_{label2}'
    se1_col = f'SE_{label1}'
    se2_col = f'SE_{label2}'
    
    # Set up labels
    if labels is None:
        labels = [label1, label2, "Both", "None"]
    
    label_map = {
        label1: labels[0],
        label2: labels[1],
        'Both': labels[2],
        'None': labels[3]
    }
    
    # Default color scheme - YOUR CUSTOM COLORS
    if color_map is None:
        color_map = {
            labels[0]: '#87CEEB',  # Light blue for males (cohort 1)
            labels[1]: '#E64B35',  # Red for females (cohort 2)
            labels[2]: '#00A087',  # Green for both
            labels[3]: '#808080'   # Grey for none
        }
    
    # Marker styles for each category
    marker_map = {
        labels[0]: 'o',  # Circle for males
        labels[1]: '^',  # Triangle for females
        labels[2]: 's',  # Square for both
        labels[3]: 'o'   # Dot for none
    }
    
    # Map categories to display labels
    merged['display_label'] = merged['category'].map(label_map)
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Check if SE columns exist
    has_se = se1_col in merged.columns and se2_col in merged.columns
    
    # Plot by category with specific markers and colors
    category_order = [labels[3], labels[1], labels[0], labels[2]]  # None, females, males, Both
    
    for cat in category_order:
        data = merged[merged['display_label'] == cat].copy()
        if len(data) > 0:
            # Plot error bars first if requested and available
            if plot_error_bars and has_se:
                ax.errorbar(
                    data[beta1_col],
                    data[beta2_col],
                    xerr=data[se1_col],
                    yerr=data[se2_col],
                    fmt='none',
                    ecolor=color_map[cat],
                    alpha=0.3,
                    linewidth=0.5,
                    zorder=1
                )
            
            # Plot points with specific marker
            ax.scatter(
                data[beta1_col], 
                data[beta2_col],
                c=color_map[cat],
                marker=marker_map[cat],
                s=s,
                label=f'{cat} (n={len(data):,})',
                alpha=alpha,
                edgecolors='black',
                linewidth=0.5,
                zorder=3
            )
    
    # Calculate and display concordance metrics
    metrics = calculate_concordance(merged, label1, label2)
    
    if metrics is not None:
        # Add regression line
        valid = merged[[beta1_col, beta2_col]].dropna()
        x_range = np.array([valid[beta1_col].min(), valid[beta1_col].max()])
        y_pred = metrics['slope'] * x_range + metrics['intercept']
        ax.plot(x_range, y_pred, 'k--', alpha=0.5, linewidth=1.5, label='Regression line', zorder=2)
        
        # Add diagonal reference line (y=x)
        lims = [
            np.min([ax.get_xlim(), ax.get_ylim()]),
            np.max([ax.get_xlim(), ax.get_ylim()])
        ]
        ax.plot(lims, lims, 'k-', alpha=0.3, linewidth=1, linestyle=':', label='y=x', zorder=2)
        
        # Add statistics text - MOVED TO TOP RIGHT
        stats_text = f"r = {metrics['r']:.3f}\n"
        stats_text += f"R² = {metrics['r2']:.3f}\n"
        stats_text += f"Sign concordance = {metrics['sign_concordance']:.1%}\n"
        stats_text += f"n = {metrics['n']:,}"
        
        ax.text(
            0.97, 0.05,  # Bottom right position
            stats_text,
            transform=ax.transAxes,
            verticalalignment='bottom',
            horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
            fontsize=10,
            zorder=5
        )
    
    # Collect SNPs to annotate
    snps_to_annotate = []
    
    # Add user-specified SNPs
    if annotate_snps is not None:
        snps_to_annotate.extend(annotate_snps)
    
    # Add most discordant variants
    if annotate_discordant > 0:
        discordant = find_most_discordant(merged, label1, label2, n=annotate_discordant)
        snps_to_annotate.extend(discordant)
        print(f"\nMost discordant variants (by |BETA difference|):")
        for snp in discordant:
            row = merged[merged['SNP'] == snp].iloc[0]
            beta_diff = abs(row[beta1_col] - row[beta2_col])
            print(f"  {snp}: β₁={row[beta1_col]:.3f}, β₂={row[beta2_col]:.3f}, |Δ|={beta_diff:.3f}")
    
    # Annotate SNPs
    if snps_to_annotate:
        texts = []
        
        for snp in snps_to_annotate:
            if snp in merged['SNP'].values:
                row = merged[merged['SNP'] == snp].iloc[0]
                x = row[beta1_col]
                y = row[beta2_col]
                if pd.notna(x) and pd.notna(y):
                    # Use SNP identifier for label
                    texts.append(ax.text(x, y, snp, fontsize=8, zorder=10))
        
        # Adjust text to avoid overlaps
        if texts:
            adjust_text(
                texts, 
                arrowprops=dict(arrowstyle='->', color='black', lw=0.5, alpha=0.7),
                zorder=10
            )
    
    # Formatting
    ax.set_xlabel(f'Effect size (β) - {label1}', fontsize=12, fontweight='bold')
    ax.set_ylabel(f'Effect size (β) - {label2}', fontsize=12, fontweight='bold')
    
    title = 'Beta-Beta Plot: Effect Size Comparison'
    if plot_error_bars and has_se:
        title += ' '
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    
    # Grid
    ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5, zorder=0)
    ax.set_axisbelow(True)
    
    # Legend
    ax.legend(
        loc='upper left',  # Changed from 'upper left'
        frameon=True, 
        fancybox=True, 
        shadow=True, 
        fontsize=9,
        framealpha=0.9
    )
    
    # Equal aspect ratio
    ax.set_aspect('equal', adjustable='box')
    
    plt.tight_layout()
    
    # Save
    if save:
        plt.savefig(save, dpi=dpi, bbox_inches='tight')
        print(f"\n💾 Plot saved to: {save}")
    
    return fig, ax, metrics


def compare_effect(
    path1,
    path2,
    snplist=None,
    drop=True,
    label=None,
    sig_level=5e-8,
    annotate_snps=None,
    annotate_discordant=5,
    plot_error_bars=True,
    investigate_dupes=False,
    return_dropped_dupes=False,
    color_map=None,
    alpha=0.6,
    s=50,
    figsize=(10, 10),
    save=None,
    dpi=300,
    return_data=False
):

    # Default labels
    if label is None:
        label = ["Cohort 1", "Cohort 2", "Both", "None"]
    
    cohort1_name = label[0]
    cohort2_name = label[1]
    
    print("="*80)
    print("BETA-BETA COMPARISON PLOT")
    print("="*80)
    
    # Investigate duplicates first if requested
    if investigate_dupes:
        # Load data without dropping
        if isinstance(path1, str):
            df1_raw = pd.read_csv(path1, sep='\t', low_memory=False)
        else:
            df1_raw = path1.copy()
        
        if isinstance(path2, str):
            df2_raw = pd.read_csv(path2, sep='\t', low_memory=False)
        else:
            df2_raw = path2.copy()
        
        investigate_duplicates(df1_raw, cohort1_name)
        investigate_duplicates(df2_raw, cohort2_name)
    
    # Load data
    if return_dropped_dupes:
        df1, dropped1 = load_and_prep_gwas(path1, cohort1_name, snplist=snplist, 
                                            drop_duplicates=drop, return_dropped=True)
        df2, dropped2 = load_and_prep_gwas(path2, cohort2_name, snplist=snplist, 
                                            drop_duplicates=drop, return_dropped=True)
        
        dropped_dupes_dict = {
            cohort1_name: dropped1,
            cohort2_name: dropped2
        }
        
        # Print summary
        if dropped1 is not None:
            print(f"\n📋 {cohort1_name} dropped duplicates: {len(dropped1):,} rows")
        if dropped2 is not None:
            print(f"📋 {cohort2_name} dropped duplicates: {len(dropped2):,} rows")
    else:
        df1 = load_and_prep_gwas(path1, cohort1_name, snplist=snplist, drop_duplicates=drop)
        df2 = load_and_prep_gwas(path2, cohort2_name, snplist=snplist, drop_duplicates=drop)
        dropped_dupes_dict = None
    
    # Merge
    merged = merge_gwas_for_comparison(df1, df2, cohort1_name, cohort2_name)
    
    # Classify by significance
    merged = classify_significance(merged, cohort1_name, cohort2_name, sig_level=sig_level)
    
    # Create plot
    fig, ax, metrics = create_beta_beta_plot(
        merged=merged,
        label1=cohort1_name,
        label2=cohort2_name,
        labels=label,
        sig_level=sig_level,
        annotate_snps=annotate_snps,
        annotate_discordant=annotate_discordant,
        plot_error_bars=plot_error_bars,
        color_map=color_map,
        alpha=alpha,
        s=s,
        figsize=figsize,
        save=save,
        dpi=dpi
    )
    
    print("\n" + "="*80)
    print("COMPARISON COMPLETE")
    print("="*80 + "\n")
    
    # Return based on flags
    result = [fig, ax, metrics]
    if return_data:
        result.append(merged)
    if return_dropped_dupes:
        result.append(dropped_dupes_dict)
    
    return tuple(result) if len(result) > 3 else (fig, ax, metrics)



In [ ]:
# Run the comparison
fig, ax, metrics, dropped_dupes = compare_effect(
    path1=all_male,
    path2=all_female,
    snplist=gp2_vars,
    drop=True,
    label=["ALL MALE", "ALL FEMALE", "Both", "None"],
    annotate_discordant=5,
    plot_error_bars=True,
    return_dropped_dupes=True,  # Get dropped duplicates
    save=f'{WORK_DIR}/betabeta_ALL_vs_GP2EUR.png'
)

# Access dropped duplicates
if dropped_dupes['ALL MALE'] is not None:
    print("\nDropped from ALL MALE:")
    print(dropped_dupes['ALL MALE'][['SNP', 'BETA', 'SE', 'P']].head(10))
    
    # Save to file
    dropped_dupes['ALL MALE'].to_csv(
        '{WORK_DIR}/ALL_MALE_dropped_duplicates.txt',
        sep='\t', index=False
    )

if dropped_dupes['ALL FEMALE'] is not None:
    print("\nDropped from ALL FEMALE:")
    print(dropped_dupes['ALL FEMALE'][['SNP', 'BETA', 'SE', 'P']].head(10))
    
    # Save to file
    dropped_dupes['ALL FEMALE'].to_csv(
        '{WORK_DIR}/ALL_FEMALE_dropped_duplicates.txt',
        sep='\t', index=False
    )

# Print concordance metrics
print(f"\nConcordance Metrics:")
print(f"  Pearson r: {metrics['r']:.4f}")
print(f"  R²: {metrics['r2']:.4f}")
print(f"  Sign concordance: {metrics['sign_concordance']:.1%}")
print(f"  Regression slope: {metrics['slope']:.4f}")
print(f"  N overlapping variants: {metrics['n']:,}")

# Show the plot
plt.show()